In [ ]:
!pip install -q transformers datasets librosa soundfile

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
my_audio = "/content/drive/MyDrive/Cap/myaudio.m4a"

#"/content/drive/MyDrive/Cap/3(gernan).mp3"
#"/content/drive/MyDrive/Cap/Mews 2.m4a"
#"/content/drive/MyDrive/Cap/How to Say How Are You in German German Lessons.mp3"

In [ ]:
import librosa

audio, sr = librosa.load(my_audio, sr=16000)

/tmp/ipykernel_3863/1607403050.py:3: UserWarning: PySoundFile failed. Trying audioread instead.
  audio, sr = librosa.load(my_audio, sr=16000)
/usr/local/lib/python3.13/dist-packages/librosa/core/audio.py:184: FutureWarning: librosa.core.audio.__audioread_load
	Deprecated as of librosa version 0.10.0.
	It will be removed in librosa version 1.0.
  y, sr_native = __audioread_load(path, offset, duration, dtype)


In [ ]:
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from datasets import load_dataset

# Load model and processor
processor = WhisperProcessor.from_pretrained("openai/whisper-base")
model = WhisperForConditionalGeneration.from_pretrained("openai/whisper-base")

model.config.forced_decoder_ids = None

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

# Test audio


In [ ]:
ds = load_dataset(
    "hf-internal-testing/librispeech_asr_dummy",
    "clean",
    split="validation"
)

sample = ds[54]["audio"]

#test audio

Generating validation split:   0%|          | 0/73 [00:00<?, ? examples/s]

In [ ]:
from IPython.display import Audio, display

display(Audio(
    sample["array"],
    rate=sample["sampling_rate"]
))

#test audio

In [ ]:
input_features = processor(
    sample["array"],
    sampling_rate=sample["sampling_rate"],
    return_tensors="pt"
).input_features

#test audio

In [ ]:
predicted_ids = model.generate(input_features)
#test audio

Transcription using a multilingual Whisper will default to language detection followed by transcription instead of translation to English. This might be a breaking change for your use case. If you want to instead always translate your audio to English, make sure to pass `language='en'`. See https://github.com/huggingface/transformers/pull/28687 for more details.


In [ ]:
transcription = processor.batch_decode(
    predicted_ids,
    skip_special_tokens=True
)

print(transcription[0])

#test audio

 He must have drawn his gun because the intruder said quickly, but that away you're being a fool. Out.


# My audio

In [ ]:
from IPython.display import Audio, display

display(Audio(audio, rate=sr))

#personal audio

In [ ]:
input_features = processor(
    audio,
    sampling_rate=sr,
    return_tensors="pt"
).input_features

In [ ]:
predicted_ids = model.generate(
    input_features,
    language="german",
    task="transcribe"
)

[transformers] Passing `generation_config` together with generation-related arguments=({'suppress_tokens', 'begin_suppress_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.


In [ ]:
transcription = processor.batch_decode(
    predicted_ids,
    skip_special_tokens=True
)

print(transcription[0])

[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer WhisperTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


 Ich bin der Zwanzek Jarehant. Mein ernahmt ist Alex, Wendu. VGT ist der.


# Best Audio to text (German)


In [ ]:
import torch
from transformers import pipeline

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Load pipeline
pipe = pipeline(
    "automatic-speech-recognition",
    model="bofenghuang/whisper-small-cv11-german",
    device=device
)

# Set German transcription
pipe.model.config.forced_decoder_ids = pipe.tokenizer.get_decoder_prompt_ids(
    language="de",
    task="transcribe"
)

Loading weights:   0%|          | 0/479 [00:00<?, ?it/s]

In [ ]:
# Set maximum generated tokens
pipe.model.generation_config.max_length = 225 + 1

In [ ]:
# Load your audio
waveform = {
    "raw": audio,
    "sampling_rate": sr
}

In [ ]:
# Run German speech recognition
german_transcription = pipe(waveform)["text"]

print("Whisper Small German:", german_transcription)

[transformers] Using custom `forced_decoder_ids` from the (generation) config. This is deprecated in favor of the `task` and `language` flags/config options.


Whisper Small German: Ich bin dreiundzwanzig Jahre alt, meiner Name ist Alex und wie geht es dir?


In [ ]:
print("Whisper Base:")
print(transcription)

print("\nWhisper Small German:")
print(german_transcription)

Whisper Base:
[' Ich bin der Zwanzek Jarehant. Mein ernahmt ist Alex, Wendu. VGT ist der.']

Whisper Small German:
Ich bin dreiundzwanzig Jahre alt, meiner Name ist Alex und wie geht es dir?


# German grammar checker

In [ ]:
import torch
from transformers import MBartForConditionalGeneration, MBart50TokenizerFast

# Load tokenizer
tokenizer = MBart50TokenizerFast.from_pretrained(
    "MRNH/mbart-german-grammar-corrector",
    src_lang="de_DE",
    tgt_lang="de_DE"
)

# Load model
grammar_model = MBartForConditionalGeneration.from_pretrained(
    "MRNH/mbart-german-grammar-corrector"
)

grammar_model.eval()

[transformers] The following generation flags are not valid and may be ignored: ['output_hidden_states']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


Loading weights:   0%|          | 0/519 [00:00<?, ?it/s]

MBartForConditionalGeneration(
  (model): MBartModel(
    (shared): MBartScaledWordEmbedding(250057, 1024, padding_idx=1)
    (encoder): MBartEncoder(
      (embed_tokens): MBartScaledWordEmbedding(250057, 1024, padding_idx=1)
      (embed_positions): MBartLearnedPositionalEmbedding(1026, 1024)
      (layers): ModuleList(
        (0-11): 12 x MBartEncoderLayer(
          (self_attn): MBartAttention(
            (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
            (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
          )
          (self_attn_layer_norm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True, bias=True)
          (activation_fn): GELUActivation()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024

In [ ]:
# Prepare Whisper transcription
inputs = tokenizer(
    german_transcription,
    return_tensors="pt"
)

# Generate corrected German text
generated_ids = grammar_model.generate(
    inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    forced_bos_token_id=tokenizer.lang_code_to_id["de_DE"],
    max_new_tokens=128,
    return_dict_in_generate=False
)

# Check generated output
print("Output type:", type(generated_ids))
print("Output shape:", generated_ids.shape)

# Convert generated tokens into text
corrected_transcription = tokenizer.batch_decode(
    generated_ids,
    skip_special_tokens=True
)[0]

print("Original Whisper Transcription:")
print(german_transcription)

print("\nCorrected German:")
print(corrected_transcription)

[transformers] Both `max_new_tokens` (=128) and `max_length`(=200) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Output type: <class 'torch.Tensor'>
Output shape: torch.Size([1, 24])
Original Whisper Transcription:
Ich bin dreiundzwanzig Jahre alt, meiner Name ist Alex und wie geht es dir?

Corrected German:
Ich bin dreiundzwanzig Jahre alt, mein Name ist Alex und wie geht es dir?


# German text to English

In [ ]:
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

tokenizer = AutoTokenizer.from_pretrained("Aparna852/de-en-translator")

model = AutoModelForSeq2SeqLM.from_pretrained("Aparna852/de-en-translator")

config.json:   0%|          | 0.00/1.30k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/847 [00:00<?, ?B/s]

source.spm: reconstructing file:   0%|          |  0.00B /  797kB            

source.spm: downloading bytes:           |  0.00B            

target.spm: reconstructing file:   0%|          |  0.00B /  768kB            

target.spm: downloading bytes:           |  0.00B            

... (3 outputs left out)

model.safetensors: reconstructing file:   0%|          |  0.00B /  296MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/288 [00:00<?, ?B/s]

In [ ]:
german_text = corrected_transcription

inputs = tokenizer(german_text, return_tensors="pt", truncation=True)

outputs = model.generate(**inputs, max_length=100)

english_text = tokenizer.decode(outputs[0], skip_special_tokens=True)

print("German:", german_text)
print("English:", english_text)

German: Ich bin dreiundzwanzig Jahre alt, mein Name ist Alex und wie geht es dir?
English: I am twenty-three years old, my name is Alex and how are you?


# Check if spoke it correctly (assessment)